### **Imports + configuration**

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

DATA_RAW = Path("..") / "data_raw"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

### **Chargement du JSON**

In [2]:
with open(DATA_RAW / "worldcup_2022.json", encoding="utf-8") as f:
    data_2022 = json.load(f)

### **Inspection de la structure**

In [3]:
print("Clés principales :", data_2022.keys())
print("Nombre de matchs :", len(data_2022["matches"]))
print("Exemple d'un match :")
data_2022["matches"][0]

Clés principales : dict_keys(['name', 'matches'])
Nombre de matchs : 64
Exemple d'un match :


{'round': 'Matchday 1',
 'date': '2022-11-20',
 'time': '19:00',
 'team1': 'Qatar',
 'team2': 'Ecuador',
 'score': {'ft': [0, 2], 'ht': [0, 2]},
 'goals1': [],
 'goals2': [{'name': 'Enner Valencia', 'minute': 16, 'penalty': True},
  {'name': 'Enner Valencia', 'minute': 31}],
 'group': 'Group A',
 'ground': 'Al Bayt Stadium, Al Khor'}

### **Construction du DataFrame brut**

In [4]:
df_2022_raw = pd.DataFrame(data_2022["matches"])
df_2022_raw.head()

,round,date,time,team1,team2,score,goals1,goals2,group,ground
0,Matchday 1,2022-11-20,19:00,Qatar,Ecuador,"{'ft': [0, 2], 'ht': [0, 2]}",[],"[{'name': 'Enner Valencia', 'minute': 16, 'pen...",Group A,"Al Bayt Stadium, Al Khor"
1,Matchday 2,2022-11-21,19:00,Senegal,Netherlands,"{'ft': [0, 2], 'ht': [0, 0]}",[],"[{'name': 'Cody Gakpo', 'minute': 84}, {'name'...",Group A,"Al Thumama Stadium, Doha"
2,Matchday 6,2022-11-25,16:00,Qatar,Senegal,"{'ft': [1, 3], 'ht': [0, 1]}","[{'name': 'Mohammed Muntari', 'minute': 78}]","[{'name': 'Boulaye Dia', 'minute': 41}, {'name...",Group A,"Al Thumama Stadium, Doha"
3,Matchday 6,2022-11-25,19:00,Netherlands,Ecuador,"{'ft': [1, 1], 'ht': [1, 0]}","[{'name': 'Cody Gakpo', 'minute': 6}]","[{'name': 'Enner Valencia', 'minute': 49}]",Group A,"Khalifa International Stadium, Al Rayyan"
4,Matchday 10,2022-11-29,18:00,Ecuador,Senegal,"{'ft': [1, 2], 'ht': [0, 1]}","[{'name': 'Moisés Caicedo', 'minute': 67}]","[{'name': 'Ismaila Sarr', 'minute': 44, 'penal...",Group A,"Khalifa International Stadium, Al Rayyan"


### **Extraction des scores**

In [5]:
"""
Extraction des scores à partir de la structure :
"score": {"ft": [home, away], ...}

Certaines entrées contiennent aussi :
- "ht" (half-time)
- "et" (extra time)
- "p"  (penalties)

Nous utilisons uniquement "ft" (full time) pour harmoniser avec 1930–2018.
"""

df_2022_raw["home_result"] = df_2022_raw["score"].apply(lambda s: s.get("ft", [None, None])[0])
df_2022_raw["away_result"] = df_2022_raw["score"].apply(lambda s: s.get("ft", [None, None])[1])

### **Extraction du stade et de la ville**

In [6]:
"""
La colonne "ground" contient une chaîne du type :
"Al Bayt Stadium, Al Khor"

Nous séparons :
- stadium = "Al Bayt Stadium"
- city    = "Al Khor"
"""

df_2022_raw["stadium"] = df_2022_raw["ground"].apply(lambda g: g.split(",")[0].strip())
df_2022_raw["city"] = df_2022_raw["ground"].apply(lambda g: g.split(",")[1].strip())

### **Construction de la colonne datetime**

In [7]:
df_2022_raw["date"] = pd.to_datetime(df_2022_raw["date"] + " " + df_2022_raw["time"])

### **Extraction du vainqueur**

In [8]:
def compute_result_2022(row):
    """
    Détermine le vainqueur d'un match de la Coupe du Monde 2022.

    Paramètres
    ----------
    row : pd.Series
        Ligne du DataFrame contenant :
        - home_result : buts de l'équipe 1
        - away_result : buts de l'équipe 2
        - team1       : nom de l'équipe 1
        - team2       : nom de l'équipe 2

    Retour
    ------
    str
        Nom de l'équipe gagnante.
        Retourne 'draw' en cas d'égalité.
    """
    if row["home_result"] > row["away_result"]:
        return row["team1"]
    elif row["home_result"] < row["away_result"]:
        return row["team2"]
    else:
        return "draw"

df_2022_raw["result"] = df_2022_raw.apply(compute_result_2022, axis=1)

### **Suppression des colonnes non scalaires**

In [9]:
cols_to_drop = ["score", "goals1", "goals2", "ground"]

df_2022_raw = df_2022_raw.drop(columns=[c for c in cols_to_drop if c in df_2022_raw.columns])

### **Construction du DataFrame final**

In [10]:
df_final_2022 = pd.DataFrame({
    "home_team": df_2022_raw["team1"],
    "away_team": df_2022_raw["team2"],
    "home_result": df_2022_raw["home_result"],
    "away_result": df_2022_raw["away_result"],
    "result": df_2022_raw["result"],
    "date": df_2022_raw["date"],
    "round": df_2022_raw["round"],
    "city": df_2022_raw["city"],
    "stadium": df_2022_raw["stadium"],
    "edition": "2022-QATAR"
})

df_final_2022.head()

,home_team,away_team,home_result,away_result,result,date,round,city,stadium,edition
0,Qatar,Ecuador,0,2,Ecuador,2022-11-20 19:00:00,Matchday 1,Al Khor,Al Bayt Stadium,2022-QATAR
1,Senegal,Netherlands,0,2,Netherlands,2022-11-21 19:00:00,Matchday 2,Doha,Al Thumama Stadium,2022-QATAR
2,Qatar,Senegal,1,3,Senegal,2022-11-25 16:00:00,Matchday 6,Doha,Al Thumama Stadium,2022-QATAR
3,Netherlands,Ecuador,1,1,draw,2022-11-25 19:00:00,Matchday 6,Al Rayyan,Khalifa International Stadium,2022-QATAR
4,Ecuador,Senegal,1,2,Senegal,2022-11-29 18:00:00,Matchday 10,Al Rayyan,Khalifa International Stadium,2022-QATAR


### **Vérifications de qualité**

#### **Inspection s'il y a des NaN**

In [11]:
df_final_2022.isna().sum()

home_team      0
away_team      0
home_result    0
away_result    0
result         0
date           0
round          0
city           0
stadium        0
edition        0
dtype: int64

#### **Inspection s'il y a des doublons**

In [12]:
df_final_2022.duplicated().sum()

np.int64(0)

#### **Vérification des types**

In [13]:
df_final_2022.dtypes

home_team                 str
away_team                 str
home_result             int64
away_result             int64
result                    str
date           datetime64[us]
round                     str
city                      str
stadium                   str
edition                   str
dtype: object

#### **Inspection s'il y a des scores négatifs**

In [14]:
df_final_2022[df_final_2022["home_result"] < 0]

,home_team,away_team,home_result,away_result,result,date,round,city,stadium,edition


#### **Inspection s'il y a des chaines mal formées**

In [15]:
df_final_2022["home_team"].unique()[:20]


<ArrowStringArray>
[       'Qatar',      'Senegal',  'Netherlands',      'Ecuador',
      'England',          'USA',        'Wales',         'Iran',
    'Argentina',       'Mexico',       'Poland', 'Saudi Arabia',
      'Denmark',       'France',      'Tunisia',    'Australia',
      'Germany',        'Spain',        'Japan',   'Costa Rica']
Length: 20, dtype: str

In [16]:
df_final_2022["city"].unique()[:20]

<ArrowStringArray>
['Al Khor', 'Doha', 'Al Rayyan', 'Lusail', 'Al Wakrah']
Length: 5, dtype: str

#### **Inspection s'il y a des dates invalides**

In [17]:
df_final_2022[df_final_2022["date"].isna()]

,home_team,away_team,home_result,away_result,result,date,round,city,stadium,edition


### **Export**

In [18]:
DATA_CLEAN = Path("..") / "data_clean"

df_final_2022.to_csv(DATA_CLEAN / "matches_2022_clean.csv", index=False)
df_final_2022.to_parquet(DATA_CLEAN / "matches_2022_clean.parquet", index=False)